# Rail Corrugation
## Model Optimisation

This notebook improves the strongest baseline model without changing the underlying feature representation.

Experiments:
- Feature selection
- Random Forest hyperparameter tuning
- Class-weight optimisation
- Alternative tree-based models

All experiments use the same stratified cross-validation folds and Macro F1 metric.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [2]:
DATA_DIR = Path("..")

features = pd.read_csv(DATA_DIR / "rail_features.csv")

X = features.drop(columns=["filename", "label"])
y = features["label"]

In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
baseline = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

base_scores = cross_val_score(
    baseline, X, y,
    cv=cv,
    scoring="f1_macro"
)

print(base_scores.round(3))
print("Mean:", base_scores.mean().round(3))

[0.725 0.652 0.779 0.756 0.666]
Mean: 0.716


## 1. Feature Selection

With only 272 recordings, unnecessary features may increase model variance.

We test whether restricting the model to the most informative features improves cross-validation performance.

In [5]:
baseline.fit(X, y)

importance = pd.Series(
    baseline.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importance)

rms_diff                 0.121946
side1_power_250_500      0.085276
rms_ratio                0.085072
side1_power_500_1000     0.056575
side2_power_0_100        0.054264
side2_rms                0.051246
side1_peak               0.047604
side2_power_250_500      0.045863
side2_std                0.041628
side2_skew               0.033092
side2_power_1000_2000    0.030058
side1_std                0.029717
side1_rms                0.028921
side1_power_1000_2000    0.028679
side1_skew               0.028288
side2_kurtosis           0.025137
side1_kurtosis           0.024657
side2_power_500_1000     0.024057
side1_power_2000_5000    0.020883
side2_dominant_freq      0.020712
side1_power_0_100        0.020519
side2_mean_abs           0.018569
side2_power_2000_5000    0.016576
side2_peak               0.015944
side1_mean_abs           0.014568
side1_dominant_freq      0.012660
side2_power_100_250      0.009246
side1_power_100_250      0.008244
dtype: float64

In [6]:
feature_results = {}

for n in [5, 10, 15, 20, 25, len(X.columns)]:
    cols = importance.head(n).index

    scores = cross_val_score(
        baseline, X[cols], y,
        cv=cv,
        scoring="f1_macro"
    )

    feature_results[n] = scores.mean()

display(pd.Series(feature_results, name="Macro F1"))

5     0.624730
10    0.695534
15    0.663511
20    0.711970
25    0.713359
28    0.693895
Name: Macro F1, dtype: float64

## 2. Random Forest Complexity

We test whether controlling tree depth and minimum leaf size improves generalisation.

In [7]:
rf_results = []

for depth in [None, 5, 10, 15]:
    for leaf in [1, 2, 4]:
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=depth,
            min_samples_leaf=leaf,
            class_weight="balanced",
            random_state=42
        )

        scores = cross_val_score(
            model, X, y,
            cv=cv,
            scoring="f1_macro"
        )

        rf_results.append({
            "depth": depth,
            "leaf": leaf,
            "mean_f1": scores.mean(),
            "std": scores.std()
        })

rf_results = pd.DataFrame(rf_results)

In [8]:
display(
    rf_results.sort_values("mean_f1", ascending=False)
)

,depth,leaf,mean_f1,std
0,NaN,1,0.715646,0.049533
9,15.0,1,0.715646,0.049533
6,10.0,1,0.715190,0.050000
3,5.0,1,0.709224,0.025807
1,NaN,2,0.693395,0.028014
7,10.0,2,0.693395,0.028014
10,15.0,2,0.693395,0.028014
4,5.0,2,0.684217,0.021796
2,NaN,4,0.669613,0.021546
5,5.0,4,0.669613,0.021546


## 3. Class Weighting

Side I is both the rarest class and the weakest-performing class in the baseline model.

We test whether increasing its training weight improves Macro F1.

In [9]:
display(y.value_counts())

label
Normal     234
Side II     24
Side I      14
Name: count, dtype: int64

In [10]:
weights = [
    {"Normal": 1, "Side I": 2, "Side II": 2},
    {"Normal": 1, "Side I": 3, "Side II": 2},
    {"Normal": 1, "Side I": 4, "Side II": 2},
    {"Normal": 1, "Side I": 5, "Side II": 2},
    {"Normal": 1, "Side I": 5, "Side II": 3}
]

In [11]:
weight_results = []

for weight in weights:
    model = RandomForestClassifier(
        n_estimators=500,
        class_weight=weight,
        random_state=42
    )

    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring="f1_macro"
    )

    weight_results.append({
        "weights": str(weight),
        "mean_f1": scores.mean(),
        "std": scores.std()
    })

weight_results = pd.DataFrame(weight_results)

In [12]:
display(
    weight_results.sort_values("mean_f1", ascending=False)
)

,weights,mean_f1,std
4,"{'Normal': 1, 'Side I': 5, 'Side II': 3}",0.669214,0.101161
0,"{'Normal': 1, 'Side I': 2, 'Side II': 2}",0.661765,0.098123
3,"{'Normal': 1, 'Side I': 5, 'Side II': 2}",0.659168,0.099012
1,"{'Normal': 1, 'Side I': 3, 'Side II': 2}",0.656667,0.096724
2,"{'Normal': 1, 'Side I': 4, 'Side II': 2}",0.651609,0.097559


## 4. Extra Trees

Extra Trees provides an alternative tree ensemble with greater randomisation than Random Forest.

This can reduce variance and may generalise differently on the small training dataset.

In [13]:
extra = ExtraTreesClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

extra_scores = cross_val_score(
    extra, X, y,
    cv=cv,
    scoring="f1_macro"
)

print(extra_scores.round(3))
print("Mean:", extra_scores.mean().round(3))
print("Std:", extra_scores.std().round(3))

[0.591 0.57  0.619 0.826 0.562]
Mean: 0.634
Std: 0.098


In [14]:
#final comparison

comparison = pd.DataFrame({
    "Baseline RF": base_scores,
    "Extra Trees": extra_scores
})

display(comparison)

,Baseline RF,Extra Trees
0,0.724755,0.591111
1,0.652482,0.569728
2,0.779487,0.619103
3,0.755871,0.826389
4,0.665633,0.561573
